1) Install libraries

In [1]:
!pip install -q -U pypdf pandas google-genai cohere groq huggingface_hub

2) Import Libraries

In [2]:
import os
import re
import json
import zipfile
from pathlib import Path
from getpass import getpass

import pandas as pd
from pypdf import PdfReader

from google.colab import files

from google import genai
import cohere
from groq import Groq
from huggingface_hub import InferenceClient

3) upload zip file

In [3]:
uploaded = files.upload()

Saving FTSP (Week 1)  .zip to FTSP (Week 1)   (1).zip


4) extract zip file

In [4]:
extract_folder = Path("project_files")
extract_folder.mkdir(exist_ok=True)

zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Files extracted successfully.\n")

for file in sorted(extract_folder.iterdir()):
    print(file.name)

Files extracted successfully.

1 DV.txt
2 DV.txt
3.txt
4.txt
5.txt
Sr. Staff Engineer, Design Verification.pdf


5) Locate resumes and job description

In [5]:
txt_files = sorted(extract_folder.glob("*.txt"))
pdf_files = sorted(extract_folder.glob("*.pdf"))

print("Resume files found:")
for f in txt_files:
    print("-", f.name)

print("\nPDF files found:")
for f in pdf_files:
    print("-", f.name)

if len(pdf_files) == 0:
    raise FileNotFoundError("No PDF job description found in the extracted folder.")

Resume files found:
- 1 DV.txt
- 2 DV.txt
- 3.txt
- 4.txt
- 5.txt

PDF files found:
- Sr. Staff Engineer, Design Verification.pdf


6) Read all text files

In [6]:
def read_text_file(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

raw_resumes = {}
for file_path in txt_files:
    raw_resumes[file_path.name] = read_text_file(file_path)

print("Loaded resumes:", len(raw_resumes))
print("\nSample preview:\n")

for name, text in raw_resumes.items():
    print("=" * 80)
    print(name)
    print(text[:500])
    print()

Loaded resumes: 5

Sample preview:

1 DV.txt
 
Top Skills
AXI
APB
CHI
Languages
English  (Native or Bilingual)
Mandarin  (Limited Working)
Malay  (Native or Bilingual)
Design Verification Engineer
Kulim, Kedah, Malaysia
Summary
Currently working as Design Verification Engineer at Skyechip Sdn
Bhd 
Graduated from University of Malaysia Perlis (UniMAP), majoring in
Microelectronic Engineering. 
Experience
SkyeChip
Design Verification Engineer
August 2022-Present (1 year 4 months)
Penang, Malaysia
Fuji Electric (M) Sdn Bhd
Manufacturing Engin

2 DV.txt
 
Top Skills
RTL Coding
RTL Verification
RTL Development
Languages
English
Staff Design Verification
Ho Chi Minh City Metropolitan Area
Experience
BOS Semiconductors
Staff Design Verification Engineer
December 2022-Present (1 year)
HCL Technologies
Technical Lead
March 2022-December 2022 (10 months)
RTL design for clock controller, power control, clock and power hardmacro
integration.
Integration system control block in cpu sub-system.
RTL 

7) Read the job description PDF

In [7]:
def read_pdf_text(pdf_path):
    reader = PdfReader(str(pdf_path))
    text = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text)

job_pdf_path = pdf_files[0]
raw_job_description = read_pdf_text(job_pdf_path)

print("Job description file:", job_pdf_path.name)
print("\nPreview:\n")
print(raw_job_description[:1500])

Job description file: Sr. Staff Engineer, Design Verification.pdf

Preview:

Sr. Staff Engineer, Design Verification 
 
Staff Engineer – Design Verification 
 
Client is looking for an experienced Staff Design Verification Engineer to join our dynamic team in 
Austin, TX. The ideal candidate will have 8-12 years of experience in the field of semiconductor 
design verification, and a proven track record of success in developing and executing verification 
plans for complex SoC designs. 
 
Responsibilities: 
 
The Design Verification Engineer will be responsible for verification of digital and mixed-signal 
designs, including systems-on-chip with multiple CPUs, and digital signal processors, security 
hardware, and other logic for IoT applications. 
 
Specific Responsibilities: 
• Ideal candidate should have demonstrated successful design verification tasks at block, sub-
system, and full-chip level. 
• Must have participated in all phases of chip development, from creating test plans, c

8) Text preprocessing

In [8]:
def preprocess_text(text):
    if not isinstance(text, str):
        text = str(text)

    # normalize line breaks / spaces
    text = text.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    # lowercase
    text = text.lower()

    # remove phone numbers
    text = re.sub(r'(\+?\d[\d\-\(\) ]{7,}\d)', ' ', text)

    # remove repeated punctuation-like noise but keep useful technical symbols
    # keep letters, digits, spaces, slash, plus, hash, hyphen, dot
    text = re.sub(r'[^a-z0-9/\+\#\-\.\s]', ' ', text)

    # remove multiple dots or dashes if noisy
    text = re.sub(r'[\.]{2,}', ' ', text)
    text = re.sub(r'[-]{2,}', ' ', text)

    # collapse spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

9) Apply preprocessing to resumes and job description

In [9]:
cleaned_resumes = {}
for name, text in raw_resumes.items():
    cleaned_resumes[name] = preprocess_text(text)

cleaned_job_description = preprocess_text(raw_job_description)

print("Cleaned job description preview:\n")
print(cleaned_job_description[:1200])

print("\n\nCleaned resume preview:\n")
for name, text in cleaned_resumes.items():
    print("=" * 80)
    print(name)
    print(text[:400])
    print()

Cleaned job description preview:

sr. staff engineer design verification staff engineer design verification client is looking for an experienced staff design verification engineer to join our dynamic team in austin tx. the ideal candidate will have 8-12 years of experience in the field of semiconductor design verification and a proven track record of success in developing and executing verification plans for complex soc designs. responsibilities the design verification engineer will be responsible for verification of digital and mixed-signal designs including systems-on-chip with multiple cpus and digital signal processors security hardware and other logic for iot applications. specific responsibilities ideal candidate should have demonstrated successful design verification tasks at block sub- system and full-chip level. must have participated in all phases of chip development from creating test plans creating testbench environment sv/uvm integrate vip s automate test env for randomize

10) Save cleaned text files

In [10]:
cleaned_folder = Path("cleaned_texts")
cleaned_folder.mkdir(exist_ok=True)

for name, text in cleaned_resumes.items():
    out_path = cleaned_folder / f"cleaned_{name}"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)

with open(cleaned_folder / "cleaned_job_description.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_job_description)

print("Cleaned files saved in:", cleaned_folder)
print("\nSaved files:")
for f in sorted(cleaned_folder.iterdir()):
    print("-", f.name)

Cleaned files saved in: cleaned_texts

Saved files:
- cleaned_1 DV.txt
- cleaned_2 DV.txt
- cleaned_3.txt
- cleaned_4.txt
- cleaned_5.txt
- cleaned_job_description.txt


11) preprocessing summary table

In [11]:
summary_rows = []

for name in raw_resumes:
    raw_text = raw_resumes[name]
    cleaned_text = cleaned_resumes[name]

    summary_rows.append({
        "file_name": name,
        "raw_char_count": len(raw_text),
        "cleaned_char_count": len(cleaned_text),
        "removed_chars": len(raw_text) - len(cleaned_text)
    })

summary_rows.append({
    "file_name": "job_description_pdf",
    "raw_char_count": len(raw_job_description),
    "cleaned_char_count": len(cleaned_job_description),
    "removed_chars": len(raw_job_description) - len(cleaned_job_description)
})

summary_df = pd.DataFrame(summary_rows)
summary_df

,file_name,raw_char_count,cleaned_char_count,removed_chars
0,1 DV.txt,2974,2883,91
1,2 DV.txt,1403,1362,41
2,3.txt,1699,1646,53
3,4.txt,4800,4692,108
4,5.txt,4376,4215,161
5,job_description_pdf,2562,2422,140


12) Create prompts

In [12]:
SYSTEM_PROMPT = """
You are an HR screening assistant.
Your task is to compare a candidate's resume with a job description.

Focus on:
1. Relevant technical skills
2. Relevant work experience
3. Overall suitability for the role

Return the answer in a clear and concise format with:
- Candidate Summary
- Matching Skills
- Missing / Weak Areas
- Suitability Score out of 10
"""

In [13]:
def build_user_prompt(job_description, candidate_name, resume_text):
    return f"""
Job Description:
{job_description}

Candidate Resume File Name:
{candidate_name}

Candidate Resume Text:
{resume_text}

Please evaluate this candidate against the job description.

Format your answer as:
Candidate Summary:
Matching Skills:
Missing / Weak Areas:
Suitability Score:
"""

13) Enter API keys

In [14]:
COHERE_API_KEY = getpass("Enter your Cohere API key: ")
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")
GROQ_API_KEY = getpass("Enter your Groq API key: ")
HF_API_KEY = getpass("Enter your Hugging Face API key: ")

Enter your Cohere API key: ··········
Enter your Gemini API key: ··········
Enter your Groq API key: ··········
Enter your Hugging Face API key: ··········


14) Set up API clients

In [15]:
# Gemini client
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Cohere client
co = cohere.ClientV2(api_key=COHERE_API_KEY)

# Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# Hugging Face client
hf_client = InferenceClient(api_key=HF_API_KEY)

15) Gemini response function

In [16]:
def get_gemini_response(system_prompt, user_prompt, model_name="gemini-2.5-flash"):
    full_prompt = f"System Prompt:\n{system_prompt}\n\nUser Prompt:\n{user_prompt}"

    response = gemini_client.models.generate_content(
        model=model_name,
        contents=full_prompt
    )

    return response.text

16) Cohere response function

In [17]:
def get_cohere_response(system_prompt, user_prompt, model_name="command-r-08-2024"):
    response = co.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    # Cohere V2 responses may return content blocks
    if hasattr(response, "message") and hasattr(response.message, "content"):
        parts = []
        for item in response.message.content:
            if hasattr(item, "text"):
                parts.append(item.text)
            elif isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
        return "\n".join(parts).strip()

    return str(response)

17. Grok Function

In [18]:
def get_groq_response(system_prompt, user_prompt, model_name="llama-3.3-70b-versatile"):
    response = groq_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

18. HuggingFace Function

In [19]:
def get_huggingface_response(
    system_prompt,
    user_prompt,
    model_name="meta-llama/Llama-3.1-70B-Instruct"
):
    response = hf_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=800
    )
    return response.choices[0].message.content

19) Test one resume first

In [20]:
test_candidate = list(cleaned_resumes.keys())[0]
test_resume = cleaned_resumes[test_candidate]

test_prompt = build_user_prompt(
    job_description=cleaned_job_description,
    candidate_name=test_candidate,
    resume_text=test_resume
)

print("Testing candidate:", test_candidate)

print("\n--- GEMINI OUTPUT ---\n")
try:
    print(get_gemini_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("Gemini Error:", e)

print("\n--- COHERE OUTPUT ---\n")
try:
    print(get_cohere_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("Cohere Error:", e)

print("\n--- GROQ OUTPUT ---\n")
try:
    print(get_groq_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("Groq Error:", e)

print("\n--- HUGGING FACE OUTPUT ---\n")
try:
    print(get_huggingface_response(SYSTEM_PROMPT, test_prompt))
except Exception as e:
    print("Hugging Face Error:", e)

Testing candidate: 1 DV.txt

--- GEMINI OUTPUT ---

**Candidate Summary:**
The candidate holds a Bachelor's degree in Microelectronic Engineering and has 1 year 4 months of experience as a Design Verification Engineer. Their "top skills" list AXI, APB, and CHI. Prior to their current DV role, their experience primarily consists of manufacturing, equipment maintenance, and various non-technical positions.

**Matching Skills:**
*   **Education:** Possesses a Bachelor of Engineering in Microelectronic Engineering, aligning with the BSEE/MSEE requirement.
*   **Architectural Knowledge (Partial):** Lists AXI, APB, and CHI as top skills, demonstrating some familiarity with AMBA protocols relevant to SoC architecture.

**Missing / Weak Areas:**
*   **Experience Gap:** Critically falls short of the required 8-12 years of experience in semiconductor design verification, having only 1 year 4 months of direct DV experience.
*   **Lack of Specific DV Expertise:** The resume provides no details on 

20) Run all resumes through models

In [21]:
results = []

for candidate_name, resume_text in cleaned_resumes.items():
    user_prompt = build_user_prompt(
        job_description=cleaned_job_description,
        candidate_name=candidate_name,
        resume_text=resume_text
    )

    print(f"Processing {candidate_name}...")

    try:
        gemini_output = get_gemini_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        gemini_output = f"Gemini Error: {e}"

    try:
        cohere_output = get_cohere_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        cohere_output = f"Cohere Error: {e}"

    try:
        groq_output = get_groq_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        groq_output = f"Groq Error: {e}"

    try:
        huggingface_output = get_huggingface_response(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        huggingface_output = f"Hugging Face Error: {e}"

    results.append({
        "candidate_name": candidate_name,
        "gemini_output": gemini_output,
        "cohere_output": cohere_output,
        "groq_output": groq_output,
        "huggingface_output": huggingface_output
    })

print("Done.")

Processing 1 DV.txt...
Processing 2 DV.txt...
Processing 3.txt...
Processing 4.txt...
Processing 5.txt...
Done.


21) View all results

In [22]:
for item in results:
    print("=" * 100)
    print("Candidate:", item["candidate_name"])

    print("\nGEMINI RESPONSE:\n")
    print(item["gemini_output"])

    print("\nCOHERE RESPONSE:\n")
    print(item["cohere_output"])

    print("\nGROQ RESPONSE:\n")
    print(item["groq_output"])

    print("\nHUGGING FACE RESPONSE:\n")
    print(item["huggingface_output"])
    print()

Candidate: 1 DV.txt

GEMINI RESPONSE:

**Candidate Summary:**
The candidate holds a Bachelor's degree in Microelectronic Engineering and has 1 year and 4 months of experience as a Design Verification Engineer. Prior experience includes roles in manufacturing engineering, equipment maintenance, sales, and delivery. The resume indicates familiarity with AXI, APB, and CHI protocols.

**Matching Skills:**
*   **Education:** Bachelor of Engineering in Microelectronic Engineering aligns with the BSEE/MSEE degree requirement.
*   **Protocols:** The candidate lists AXI and APB in their top skills, which are key components of the AMBA architecture (AXI/AHB/APB) required by the job description.
*   **Current Role:** Currently employed as a Design Verification Engineer, indicating an entry into the field.

**Missing / Weak Areas:**
*   **Experience Level:** Significantly falls short of the required 8-12 years of experience in semiconductor design verification, with only 1 year 4 months in a relev

### **Recomendation Section**

22. Score Extraction

In [23]:
def extract_score(text):
    """
    Extracts a score out of 10 from model output.
    Handles cases like:
    7/10
    7.5/10
    Suitability Score: 6/10
    """
    if not isinstance(text, str):
        return None

    match = re.search(r'(\d+(?:\.\d+)?)\s*/\s*10', text)
    if match:
        return float(match.group(1))

    return None

23. Score Table

In [24]:
score_rows = []

for item in results:
    score_rows.append({
        "candidate_name": item["candidate_name"],
        "gemini_score": extract_score(item["gemini_output"]),
        "cohere_score": extract_score(item["cohere_output"]),
        "groq_score": extract_score(item["groq_output"]),
        "huggingface_score": extract_score(item["huggingface_output"]),
    })

scores_df = pd.DataFrame(score_rows)
scores_df

,candidate_name,gemini_score,cohere_score,groq_score,huggingface_score
0,1 DV.txt,2.0,6.0,4.0,2.0
1,2 DV.txt,5.0,7.0,8.0,6.5
2,3.txt,8.5,6.0,7.5,7.5
3,4.txt,3.0,5.0,NaN,4.0
4,5.txt,2.0,6.0,4.0,4.0


24. Average Score

In [25]:
score_columns = ["gemini_score", "cohere_score", "groq_score", "huggingface_score"]

scores_df["average_score"] = scores_df[score_columns].mean(axis=1, skipna=True)
scores_df = scores_df.sort_values(by="average_score", ascending=False)

scores_df

,candidate_name,gemini_score,cohere_score,groq_score,huggingface_score,average_score
2,3.txt,8.5,6.0,7.5,7.5,7.375
1,2 DV.txt,5.0,7.0,8.0,6.5,6.625
3,4.txt,3.0,5.0,NaN,4.0,4.000
4,5.txt,2.0,6.0,4.0,4.0,4.000
0,1 DV.txt,2.0,6.0,4.0,2.0,3.500


25. Each Model's Top Reccomendation

In [26]:
# model_recommendations = {}

# for col in score_columns:
#     valid_df = scores_df.dropna(subset=[col])
#     if not valid_df.empty:
#         top_candidate = valid_df.sort_values(by=col, ascending=False).iloc[0]["candidate_name"]
#         model_recommendations[col] = top_candidate

# print("Top recommendation by each LLM:\n")
# for model_name, candidate in model_recommendations.items():
#     print(f"{model_name}: {candidate}")

26. Majority Vote

In [27]:
# from collections import Counter

# votes = list(model_recommendations.values())
# vote_counter = Counter(votes)

# print("Majority vote result:\n")
# for candidate, count in vote_counter.items():
#     print(f"{candidate}: {count} vote(s)")

# majority_best_candidate = vote_counter.most_common(1)[0][0]
# print("\nBest resume based on majority vote:", majority_best_candidate)

27. Final Best Candidate

In [28]:
# best_average_candidate = scores_df.iloc[0]["candidate_name"]
# print("Best resume based on average score:", best_average_candidate)

# if majority_best_candidate == best_average_candidate:
#     final_best_candidate = majority_best_candidate
# else:
#     # if majority and average differ, prefer average score
#     final_best_candidate = best_average_candidate

# print("FINAL RECOMMENDED RESUME:", final_best_candidate)

ALL COMBINED RESULTS TESTING

In [29]:
from collections import Counter

score_columns = ["gemini_score", "cohere_score", "groq_score", "huggingface_score"]

# Find top candidate for each model
model_recommendations = {}

for col in score_columns:
    valid_df = scores_df.dropna(subset=[col])
    if not valid_df.empty:
        top_candidate = valid_df.sort_values(by=col, ascending=False).iloc[0]["candidate_name"]
        model_recommendations[col] = top_candidate

print("Top recommendation by each LLM:\n")
for model_name, candidate in model_recommendations.items():
    print(f"{model_name}: {candidate}")

# Majority vote
votes = list(model_recommendations.values())
vote_counter = Counter(votes)

print("\nMajority vote result:\n")
for candidate, count in vote_counter.items():
    print(f"{candidate}: {count} vote(s)")

majority_best_candidate = vote_counter.most_common(1)[0][0]

# Best candidate by average score
best_average_candidate = scores_df.sort_values(by="average_score", ascending=False).iloc[0]["candidate_name"]

print("\nBest resume based on average score:", best_average_candidate)

# Final decision
if majority_best_candidate == best_average_candidate:
    final_best_candidate = majority_best_candidate
else:
    final_best_candidate = best_average_candidate

print("\nFINAL RECOMMENDED RESUME:", final_best_candidate)

Top recommendation by each LLM:

gemini_score: 3.txt
cohere_score: 2 DV.txt
groq_score: 2 DV.txt
huggingface_score: 3.txt

Majority vote result:

3.txt: 2 vote(s)
2 DV.txt: 2 vote(s)

Best resume based on average score: 3.txt

FINAL RECOMMENDED RESUME: 3.txt


28) Save results to CSV

In [30]:
results_df = pd.DataFrame(results)
results_df.to_csv("week1_2_llm_screening_results.csv", index=False)

print("Saved: week1_2_llm_screening_results.csv")
results_df

Saved: week1_2_llm_screening_results.csv


,candidate_name,gemini_output,cohere_output,groq_output,huggingface_output
0,1 DV.txt,**Candidate Summary:**\nThe candidate holds a ...,Candidate Summary: \nThis candidate is a Desig...,Candidate Summary:\nThe candidate is a design ...,Candidate Summary:\nThe candidate is a Design ...
1,2 DV.txt,**Candidate Summary:**\nThe candidate presents...,Candidate Summary:\nThis candidate has a stron...,Candidate Summary:\nThe candidate has around 1...,Candidate Summary:\nThe candidate has approxim...
2,3.txt,**Candidate Summary:**\nAn experienced Design ...,Candidate Summary:\nThis candidate has experie...,Candidate Summary:\nThe candidate has approxim...,Candidate Summary:\nThe candidate is an experi...
3,4.txt,**Candidate Summary:**\nThe candidate holds an...,Candidate Summary: \nThis candidate has a stro...,Candidate Summary:\nThe candidate has a master...,Candidate Summary:\nThe candidate has a backgr...
4,5.txt,**Candidate Summary:**\nThe candidate is an ex...,Candidate Summary:\nThe candidate is an experi...,Candidate Summary:\nThe candidate is an experi...,Candidate Summary:\nThe candidate is an experi...


29) Save results to JSON too

In [31]:
with open("week1_2_llm_screening_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved: week1_2_llm_screening_results.json")

Saved: week1_2_llm_screening_results.json


30) Download outputs from Colab

In [32]:
files.download("week1_2_llm_screening_results.csv")
files.download("week1_2_llm_screening_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

preprcessing testing

In [33]:
candidate = list(raw_resumes.keys())[0]

print("RAW TEXT:\n")
print(raw_resumes[candidate][:500])

print("\n" + "="*60 + "\n")

print("CLEANED TEXT:\n")
print(cleaned_resumes[candidate][:500])

RAW TEXT:

 
Top Skills
AXI
APB
CHI
Languages
English  (Native or Bilingual)
Mandarin  (Limited Working)
Malay  (Native or Bilingual)
Design Verification Engineer
Kulim, Kedah, Malaysia
Summary
Currently working as Design Verification Engineer at Skyechip Sdn
Bhd 
Graduated from University of Malaysia Perlis (UniMAP), majoring in
Microelectronic Engineering. 
Experience
SkyeChip
Design Verification Engineer
August 2022-Present (1 year 4 months)
Penang, Malaysia
Fuji Electric (M) Sdn Bhd
Manufacturing Engin


CLEANED TEXT:

top skills axi apb chi languages english native or bilingual mandarin limited working malay native or bilingual design verification engineer kulim kedah malaysia summary currently working as design verification engineer at skyechip sdn bhd graduated from university of malaysia perlis unimap majoring in microelectronic engineering. experience skyechip design verification engineer august 2022-present 1 year 4 months penang malaysia fuji electric m sdn bhd manufacturing

In [34]:
for name in raw_resumes:
    raw_len = len(raw_resumes[name])
    clean_len = len(cleaned_resumes[name])

    print(name)
    print("Raw length:", raw_len)
    print("Cleaned length:", clean_len)
    print("Removed characters:", raw_len - clean_len)
    print()

1 DV.txt
Raw length: 2974
Cleaned length: 2883
Removed characters: 91

2 DV.txt
Raw length: 1403
Cleaned length: 1362
Removed characters: 41

3.txt
Raw length: 1699
Cleaned length: 1646
Removed characters: 53

4.txt
Raw length: 4800
Cleaned length: 4692
Removed characters: 108

5.txt
Raw length: 4376
Cleaned length: 4215
Removed characters: 161



In [35]:
import re

for name, text in cleaned_resumes.items():
    emails = re.findall(r'\S+@\S+', text)

    if emails:
        print(name, "Still has email:", emails)
    else:
        print(name, "No emails found")

1 DV.txt No emails found
2 DV.txt No emails found
3.txt No emails found
4.txt No emails found
5.txt No emails found
